# Lab workbook 5 - Growth and Accessibility

In [1]:
# Setting up:
#%pip install osmnx mapclassify folium overturemaps

import os
import math
import getpass
import requests
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.ops import unary_union

print("Ready! OSMnx version", ox.__version__)
print("Files in data/:", sorted(os.listdir('Challenge_7/data')))

Ready! OSMnx version 1.5.0
Files in data/: ['.ipynb_checkpoints', 'barcelona_gracia_buildings.gpkg', 'barcelona_gracia_walk.graphml', 'paris_buildings.gpkg', 'paris_overture_places.gpkg', 'paris_parks.gpkg', 'paris_walk.graphml']


## Challenge 7: A 15-minute neighbourhood analysis

You now have all the tools to study accessibility on your own. Choose **one neighbourhood** from the list below (or another one you know well), and answer:

> **Is this a 15-minute neighbourhood? Who has good access to parks and everyday services, and where are the gaps?**

### Choose a neighbourhood

These areas have very complete OpenStreetMap data and download quickly:

| City | Neighbourhood | `CENTRE` (latitude, longitude) |
|---|---|---|
| Paris | Le Marais | `(48.8590, 2.3620)` |
| Barcelona | Gràcia | `(41.4030, 2.1560)` |
| Amsterdam | De Pijp | `(52.3540, 4.8930)` |

### What to do

Create new cells below and reuse the code from this lab. You will need:

1. **Data** (Parts 3 and 6): within `DIST = 2000` metres of your centre, download
   - the walk network with `ox.graph_from_point(...)`
   - parks (`{"leisure": "park"}`) and buildings (`{"building": True}`) with `ox.features_from_point(...)`
   - services from **Overture** with `overturemaps.record_batch_reader(...)`: at least **two** types, e.g. food shops, healthcare and schools
2. **Isochrones** (Part 5): draw the 5, 10 and 15-minute walks from the centre of your neighbourhood. Show the services inside, and count how many of each type each isochrone contains.
3. **2SFCA** (Part 6): calculate park space per building within a 15-minute walk, and map it for homes within **800 m** of the centre.
4. **A summary table** with at least: number of parks, buildings and homes; median park m² per building; % of homes with no park within 15 minutes; services within a 15-minute walk of the centre.
5. **Your Conclusions**: a markdown cell of **100 to 150 words** answering the question above. Use your maps and numbers as evidence, and mention at least **one limitations** of your analysis.
6. If you use AI, describe where you have used it and how. You have all the code you need in this Lab, so detecting new code will be easy, so avoid the hassle.

**Challenge 7**:
I have chosen to do the 15 minute neighbourhood analysis for Le Marais, Paris. The analysis is using the walking network,park polygons,building footprints,and Overture places data so that we can look at the access to everyday services and park space in Le Marais, Paris, then i am applying the isochrone and 2SFCA methods from lab 5.

In [2]:
# Starting with using Le Marais, Paris, as suggested in the Challenge 7 instructions
# I am defining the centre and keeping the same walking speed and 15-minute catchment from Lab 5 - as the larger 2 km data area helps include parks and buildings outside the central neighbourhood
#code from: lab 5 cells 38,84,93

CITY = "Le Marais, Paris"
CENTRE = (48.8590, 2.3620)      # latitude, longitude
DIST = 2000                     # metres: intended download distance from Challenge 7
CATCHMENT = 1200                # metres: a 15-minute walk
WALK_SPEED = 80                 # metres per minute

**Reproducability note and change from live downloads:**
I initially attempted to download the Paris walking network using ox.graph_from_point() as specified in Challenge 7 - however the request failed because the connection to the Overpass server timed out. Hence, I have decided to use the pre-downloaded Paris walking network, parks, buildings and Overture places supplied with the course materials. 
These datasets are used throughout the analysis so that the results can be reproduced without relying on live download availability.

In [3]:
#Frist i am loading the supplied Paris walking network using the taught load_graphml method from lab 5
# Since the live download failed, so I am using the saved dataset instead
# Then i am projecting the network so that walking distances can be measured in metres as we did in Lab 5 Part 7
#code from lab 5 cells 63,84

G_city = ox.load_graphml("Challenge_7/data/paris_walk.graphml")
G_city = ox.project_graph(G_city).to_undirected()
crs = G_city.graph["crs"]

print("Walking network:", len(G_city), "junctions")
print("Projected CRS:", crs)

Walking network: 16155 junctions
Projected CRS: +proj=utm +zone=31 +ellps=WGS84 +datum=WGS84 +units=m +no_defs +type=crs


In [4]:
#then i need to load the Paris park and building datasets
# following the lab 5 Part 7 example and using the network CRS for both layers- then the park area is calculated in m^2 and buildings will be used as a proxy for demand like lab 5 examples
#code from lab 5 cell 84
parks = gpd.read_file("Challenge_7/data/paris_parks.gpkg").to_crs(crs)
parks["area_m2"] = parks.area

buildings = gpd.read_file("Challenge_7/data/paris_buildings.gpkg").to_crs(crs)

print(len(parks), "parks and", len(buildings), "buildings in", CITY)

167 parks and 23351 buildings in Le Marais, Paris


In [5]:
# then loading the supplied Paris Overture places using the method from Lab 5 Part 4
# and also checking the categories first because different cities can have different basic_category labels
#code from lab 5 cells 50,52

places = gpd.read_file("Challenge_7/data/paris_overture_places.gpkg").to_crs(crs)

print(len(places), "Overture places in", CITY)
places["basic_category"].value_counts().head(25)


110192 Overture places in Le Marais, Paris


basic_category
restaurant                            10719
fashion_and_apparel_store              7867
professional_service                   6520
corporate_or_business_office           3873
personal_or_beauty_service             3494
behavioral_or_mental_health_clinic     3385
media_service                          3268
design_service                         3064
financial_service                      2638
attorney_or_law_firm                   2534
home_service                           2454
real_estate_service                    2233
food_and_beverage_store                2033
bar                                    1898
specialized_health_care                1858
casual_eatery                          1834
hotel                                  1801
social_or_community_service            1768
wellness_service                       1602
hardware_home_and_garden_store         1470
travel_service                         1411
diagnostics_imaging_or_lab_service     1273
art_gallery      

In [6]:
#next i need to be selecting two types of everyday services from the Overture data
# Then using str.contains and confidence filtering method from Part 4
# I chose convenience stores and pharmacies after checking the available Paris categories
#code from lab 5 cell 54 and also challenge 3.2:

is_food = places["basic_category"].str.contains("grocery|supermarket|convenience", na=False)
overture_shops = places[is_food & (places["confidence"] >= 0.5)].copy()

is_pharmacy = places["basic_category"].str.contains("pharmacy", na=False)
overture_pharmacies = places[is_pharmacy & (places["confidence"] >= 0.5)].copy()

print("Convenience stores:", len(overture_shops))
print(overture_shops["basic_category"].value_counts())
print()
print("Pharmacies:", len(overture_pharmacies))
print(overture_pharmacies["basic_category"].value_counts())


Convenience stores: 282
basic_category
convenience_store    282
Name: count, dtype: int64

Pharmacies: 493
basic_category
pharmacy_and_drug_store    493
Name: count, dtype: int64


In [7]:
#Next is converting the neighbourhood centre from latitude/longitude to the same CRS as the walking network
# Using the the GeoSeries.from_xy method used in Lab 5 Part 5 as a guide
# Then finding the nearest network junction, as demonstrated in the isochrone example in Part 6
#code from lab 5 cells 65,75
centre = gpd.GeoSeries.from_xy([CENTRE[1]], [CENTRE[0]], crs="EPSG:4326").to_crs(crs)

home = ox.distance.nearest_nodes(G_city, centre.x.iloc[0], centre.y.iloc[0])

print("Nearest network junction:", home)

Nearest network junction: 11051263286


In [8]:
#Then uing the original isochrone function provided in Lab 5 Part 6 - because it identifies the reachable street network using distance and buffers the streets by 50 m
# then this produces the walking catchment polygons used for the maps and service counts
#code from lab 5 cell 73

def isochrone(G, start_node, metres):
    reachable = nx.ego_graph(G, start_node, radius=metres, distance="length")
    streets = ox.graph_to_gdfs(reachable, nodes=False)
    return unary_union(list(streets.buffer(50)))


In [ ]:
#Next to do is mapping 5, 10 and 15-minute walks from the neighbourhood centre - using the isochrone plotting code from Lab 5 Part 6 as a guide, I am displaying convenience stores and pharmacies within the 15-minute catchment
#code from lab 5 cell75
city_nodes, city_edges = ox.graph_to_gdfs(G_city)

fig, ax = plt.subplots(figsize=(9, 9))
city_edges.plot(ax=ax, color="lightgrey", linewidth=0.4)

for minutes, colour in [(15, "#cfe0f0"), (10, "#7fb0dd"), (5, "#00539B")]:
    shape = isochrone(G_city, home, minutes * WALK_SPEED)
    gpd.GeoSeries([shape], crs=crs).plot(ax=ax, color=colour, alpha=0.8)

iso15 = isochrone(G_city, home, 15 * WALK_SPEED)

food_inside = overture_shops[overture_shops.within(iso15)]
pharmacies_inside = overture_pharmacies[overture_pharmacies.within(iso15)]

food_inside.plot(ax=ax, color="#608738", markersize=25, label="Convenience stores",
                 edgecolor="black", linewidth=0.3)
pharmacies_inside.plot(ax=ax, color="#C22A22", markersize=25, label="Pharmacies",
                       edgecolor="black", linewidth=0.3)

ax.legend(loc="lower left")
ax.set_xlim(iso15.bounds[0] - 300, iso15.bounds[2] + 300)
ax.set_ylim(iso15.bounds[1] - 300, iso15.bounds[3] + 300)
ax.set_title("Le Marais, Paris: 5, 10 and 15-minute walks with everyday services")
ax.set_axis_off()
plt.show()


Figure 11: Showing the walking accessibility to everyday services from the centre of Le Marais, Paris, showing the 5-min (dark blue), 10-min (medium blue) and 15-min (light blue) walking isochrones. The green points represent convenience stores and red points represent pharmacies from the Overture dataset. The walking catchments were calculated using the street network and an assumed walking speed of 80 m/min.

![le_marais_walking_services](screenshots/le_marais_walking_services.png)

In [10]:
#Since challenge 7 asks for the number of each service within all three walking catchments - i will be using the part 6 example as a guide as it counts services within one isochrone using .within()
# then i am repeating the same method for 5, 10 and 15 minutes
#code from lab 5 75
for minutes in [5, 10, 15]:
    shape = isochrone(G_city, home, minutes * WALK_SPEED)

    food_inside = overture_shops[overture_shops.within(shape)]
    pharmacies_inside = overture_pharmacies[overture_pharmacies.within(shape)]

    print(f"{minutes}-minute walk:")
    print("Convenience stores:", len(food_inside))
    print("Pharmacies:", len(pharmacies_inside))
    print()


5-minute walk:
Convenience stores: 3
Pharmacies: 4

10-minute walk:
Convenience stores: 18
Pharmacies: 28

15-minute walk:
Convenience stores: 38
Pharmacies: 71



**Table 1.** Number of mapped convenience stores and pharmacies within each walking isochrone from the centre of Le Marais, calculated using the Overture places dataset

| Walking time | Convenience stores | Pharmacies |
|---|---:|---:|
| 5 minutes | 3 | 4 |
| 10 minutes | 18 | 28 |
| 15 minutes | 38 | 71 |

In [11]:
#Now i am starting the 2SFCA park accessibility analysis using Lab 5 Part 7 as help
# First i am assigning each park and building to its nearest junction on the Paris walking network - and the number of buildings at each junction is then used as a proxy for demand
#code from lab 5 cell 86
parks["node"] = ox.distance.nearest_nodes(G_city, parks.representative_point().x, parks.representative_point().y)
buildings["node"] = ox.distance.nearest_nodes(G_city, buildings.representative_point().x, buildings.representative_point().y)

demand = buildings["node"].value_counts()        # junction -> number of buildings
demand.head()


node
3135440151     39
11390701117    34
25581588       29
12178151018    25
256619150      24
Name: count, dtype: int64

In [12]:
#this is Step 1 of the 2SFCA method, copied from Lab 5 Part 7, cell 88 - for each park i am finding junctions within a 1200 m walking distance using Dijkstra's method
# Dividing park area by the number of buildings in its catchment to estimate shared park supply
#code from lab 5 cell 88

reach = {}          # which junctions can walk to each park
ratio = {}          # park area divided by the buildings that can reach it

for park_id, park in parks.iterrows():
    nearby = nx.single_source_dijkstra_path_length(G_city, park["node"], cutoff=CATCHMENT, weight="length")
    buildings_nearby = sum(demand.get(node, 0) for node in nearby)
    reach[park_id] = nearby
    ratio[park_id] = park["area_m2"] / buildings_nearby if buildings_nearby > 0 else 0

parks["m2_per_building"] = pd.Series(ratio)
parks.sort_values("m2_per_building", ascending=False)[["name", "area_m2", "m2_per_building"]].head(10).round(0)


,name,area_m2,m2_per_building
5,Jardin des Plantes,225847.0,143.0
112,Parc des Buttes-Chaumont,251507.0,125.0
96,Jardin du Luxembourg,217677.0,82.0
134,Jardin de Reuilly - Paul-Pernin,14202.0,14.0
27,Jardin Tino Rossi - Musée de la Sculpture en P...,28555.0,12.0
102,Parc de Belleville,30863.0,11.0
23,Jardin des Grands-Explorateurs Marco Polo et R...,10645.0,10.0
103,Jardin Nelson Mandela,38173.0,8.0
15,Jardin Villemin - Mahsa Jîna Amini,18869.0,8.0
164,Square Marie Curie,6976.0,8.0


In [13]:
#this is step 2 of the 2SFCA method, copied from Lab 5 Part 7
# I am summing the park-space ratios of all parks that can be reached from each junction
# Then selecting junctions associated with buildings, as a proxy for home locations
#code from lab 5 cell 90

city_nodes, city_edges = ox.graph_to_gdfs(G_city)
park_access = {node: 0.0 for node in G_city.nodes}

for park_id, nearby in reach.items():
    for node in nearby:
        park_access[node] += ratio[park_id]

city_nodes["park_access"] = pd.Series(park_access)

homes_city = city_nodes[city_nodes.index.isin(demand.index)]       # junctions that have buildings
homes_city["park_access"].describe().round(0)


count    7988.0
mean       50.0
std        55.0
min         2.0
25%        18.0
50%        23.0
75%        42.0
max       263.0
Name: park_access, dtype: float64

In [14]:
# then since sallenge 7 asks for park accessibility to be reported for homes within 800 m of the centre - i need to be using the .buffer() and .within() methods demonstrated in Lab 5 Part 5
# This only changes which home junctions are reported; the 1200 m 2SFCA catchment stays unchanged
#code from lab 5 cell 65

neighbourhood = centre.buffer(800).iloc[0]

homes_city = homes_city[homes_city.within(neighbourhood)].copy()

print(len(homes_city), "home junctions within 800 m of the centre")
homes_city["park_access"].describe().round(0)


1252 home junctions within 800 m of the centre


count    1252.0
mean       20.0
std         6.0
min         7.0
25%        16.0
50%        19.0
75%        23.0
max        41.0
Name: park_access, dtype: float64

In [ ]:
# Next is mapping the 2SFCA scores for the central Le Marais neighbourhood
# The original point classification and park plotting come from Lab 5 Part 7 - and then adding the 800 m neighbourhood boundary and zoom using the taught methods
#code from lab 5 cell 91 and boundary and zoom methods from cell 70 and 79

fig, ax = plt.subplots(figsize=(9, 9))
city_edges.plot(ax=ax, color="lightgrey", linewidth=0.4)
parks.plot(ax=ax, color="#a8d5a2")
homes_city.sort_values("park_access").plot(ax=ax, column="park_access", cmap="Greens", scheme="quantiles", k=5,
                                           markersize=8, legend=True,
                                           legend_kwds={"title": "Park m² per building (2SFCA)", "loc": "lower left"})

gpd.GeoSeries([neighbourhood], crs=crs).boundary.plot(ax=ax, color="grey")

west, south, east, north = neighbourhood.buffer(400).bounds
ax.set_xlim(west, east)
ax.set_ylim(south, north)

ax.set_title(f"{CITY}: park accessibility within 15 minutes (2SFCA)")
ax.set_axis_off()
plt.show()

print(f"{(homes_city['park_access'] == 0).mean():.0%} of home junctions have no park within a 15-minute walk")

Figure 12: The estimated accessible park space per building for building-associated street junctions within 800 m of the centre of Le Marais - using the two-step floating catchment area (2SFCA) method and a 1,200 m walking catchment. The darker green points represent a higher park accessibility, and the lighter green points show lower accessibility. Then the light green polygons represent mapped parks, and the grey circle outlines the 800 m neighbourhood reporting area.

![le_marais_parks_acessibility](screenshots/le_marais_parks.png)

In [16]:
#lastly is creating the summary table required in Challenge 7 - i am using the pd.DataFrame method demonstrated in Lab 5 Part 4
#Combining the dataset counts, central 2SFCA statistics and services within the 15-minute isochrone:
#code from lab 5 cell 57 and the specific summary table requirements for challenge 7

summary = pd.DataFrame({
    "Measure": [
        "Number of parks in supplied dataset",
        "Number of buildings in supplied dataset",
        "Home junctions within 800 m",
        "Median park m² per building (2SFCA)",
        "% of home junctions with no park access",
        "Convenience stores within a 15-minute walk",
        "Pharmacies within a 15-minute walk"
    ],
    "Result": [
        len(parks),
        len(buildings),
        len(homes_city),
        homes_city["park_access"].median(),
        (homes_city["park_access"] == 0).mean() * 100,
        len(overture_shops[overture_shops.within(iso15)]),
        len(overture_pharmacies[overture_pharmacies.within(iso15)])
    ]
})

summary.round(1)


,Measure,Result
0,Number of parks in supplied dataset,167.0
1,Number of buildings in supplied dataset,23351.0
2,Home junctions within 800 m,1252.0
3,Median park m² per building (2SFCA),19.4
4,% of home junctions with no park access,0.0
5,Convenience stores within a 15-minute walk,38.0
6,Pharmacies within a 15-minute walk,71.0


**Table 2.** Summary of the Le Marais neighbourhood analysis: showing the number of parks and buildings in the supplied dataset, the number of building-associated junctions within the central 800 m reporting area, and the park-accessibility and everyday-service results.

The median 2SFCA score and percentage with no park access refer to the 1,252 selected junctions, and the convenience-store and pharmacy counts represent locations within a 15-minute walk of the neighbourhood centre.

### Description
**Challenge 7 description and insights**
For this challenge, I chose Le Marais in Paris (from the options) to look into if it functions as a '15-minute neighbourhood' or not i did this using the network accessibility and 2SFCA methods demonstrated in Lab 5. Then I used the supplied Paris walking network, park polygons, building footprints and Overture places data - since the the live OSMnx download timed out multiple times. 
Firstly, I used the walking network to create the 5, 10 and 15 min isochrones from the neighbourhood centre (using an assuming a walking speed of 80 m/min). Then i needed to identify and count the convenience stores and pharmacies within each catchment and i did this using the Overture data and a confidence threshold of 0.5 and the methods again from lab 5.
Next up, I applied the two-step floating catchment area (2SFCA) method from lab 5 to estimate the accessible park space per building -this included having to calculate the area of each park relative to the number of buildings within a 1,200 m network walking catchment, and then combining the accessible park-space ratios for each junction, then I focused the final results on building-associated junctions within 800 m of the neighbourhood centre using lab 5 methods and workflow to support this thought proccess as I went.
All this combined, meant that i could compare the access to everyday services with the spatial distribution of available park space - but while also taking account of both walking distance and the number of buildings sharing access to each park.

### Conclusion
To conclude from this challenge 7 exercide, the results suggest that the centre of Le Marais performs pretty well as a '15-minute neighbourhood', although this accessibility does vary spatially. The 15 min walking isochrone from the neighbourhood centre contains 38 convenience stores and 71 pharmacies - this shows good access to these everyday services for residents. The 2SFCA resuls showed that all of the  1,252 selected home-proxy junctions had some park accessibility -  with a median value of 19.4 m² per building. However, it should be noted that accessibility was not equally distributed asthere were lower scores in parts of the northern and eastern neighbourhood.
Also to note from this challenge, is that there are limitations to the analysis - this is because the buildings were used as a proxy for population demand so differences in occupancy weren't considered, also the isochrones use 50 m buffers around reachable streets meaning that service accessibility is more of an approximate value rather than an exact walking route to every location.